In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# ==============================================================================
# ПАРАМЕТРИ ВАРІАНТА 6
# Місто: Чернігів
# ==============================================================================
VARIANT = 6
CITY = "Чернігів"
MONTHLY_TEMP = [-5, -4, 1, 9, 15, 18, 20, 19, 13, 7, 1, -3]
MONTHS = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
ALPHA = 0.05

np.random.seed(VARIANT)

# Генерація вихідного набору даних
rows = []
for month, temp in zip(MONTHS, MONTHLY_TEMP):
    heating_days = np.clip(18 - temp + np.random.normal(0, 1.5), 0, 30)
    heating_days = round(heating_days)
    cost = 0.15 * heating_days + np.random.normal(0, 1.0)
    cost = round(max(cost, 0), 1)
    rows.append({
        "місто": CITY,
        "місяць": month,
        "температура": temp,
        "опалювальні_дні": heating_days,
        "витрати_на_опалення": cost,
    })

climate = pd.DataFrame(rows)

print(f"=== ПРАКТИКА 16: Варіант {VARIANT} ({CITY}) ===")
print(climate.to_string(index=False))

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. Діаграма розсіювання
# ------------------------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.scatter(climate["температура"], climate["витрати_на_опалення"], color="crimson", edgecolors="black", s=70)
plt.title(f"Залежність витрат на опалення від температури ({CITY})", fontsize=12)
plt.xlabel("Середньомісячна температура, °C", fontsize=10)
plt.ylabel("Витрати на опалення, тис. грн", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("scatter_plot_v6.png")
plt.close()

print("\n=== Завдання 1: Діаграма розсіювання ===")
print("Графік збережено як 'scatter_plot_v6.png'.")
print("Опис хмари точок: Спостерігається чітко виражена від'ємна (негативна) лінійна тенденція.")
print("При збільшенні температури витрати на опалення зменшуються. При високих температурах (влітку)")
print("витрати знижуються до 0, що формує характерне 'сплющення' (плато) на рівні нульових витрат.")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Коефіцієнти Пірсона і Спірмена
# ------------------------------------------------------------------------------
print("\n=== Завдання 2: Коефіцієнти Пірсона і Спірмена ===")

r_res = pearsonr(climate["температура"], climate["витрати_на_опалення"])
rho_res = spearmanr(climate["температура"], climate["витрати_на_опалення"])

r_stat, r_p = r_res.statistic, r_res.pvalue
rho_stat, rho_p = rho_res.statistic, rho_res.pvalue
ci = r_res.confidence_interval(confidence_level=0.95)

print(f"Коефіцієнт Пірсона (r):   {r_stat:.4f}, p-value: {r_p:.4e}")
print(f"95% Довірчий інтервал для r: [{ci.low:.4f}, {ci.high:.4f}]")
print(f"Коефіцієнт Спірмена (rho): {rho_stat:.4f}, p-value: {rho_p:.4e}")

task2_text = f"""
[Висновки до Завдання 2]
1. Гіпотези H0 (кореляція = 0):
   - Для Пірсона: p-value ({r_p:.4e}) < {ALPHA} -> H0 відхиляємо.
   - Для Спірмена: p-value ({rho_p:.4e}) < {ALPHA} -> H0 відхиляємо.
   Зв'язок є статистично значущим, від'ємним і надзвичайно сильним.
2. Порівняння r і rho:
   Обидва значення є близькими (r ≈ {r_stat:.3f}, rho ≈ {rho_stat:.3f}), що свідчить про близькість
   зв'язку до монотонного та лінійного на цьому інтервалі.
3. Інтерпретація 95% довірчого інтервалу для r [{ci.low:.3f}; {ci.high:.3f}]:
   З ймовірністю 95% істинне значення коефіцієнта кореляції Пірсона в генеральній сукупності
   знаходиться в межах від {ci.low:.3f} до {ci.high:.3f}. Інтервал не містить 0, що підтверджує значущість.
"""
print(task2_text)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. Кореляційна матриця трьох змінних
# ------------------------------------------------------------------------------
print("=== Завдання 3: Кореляційні матриці (3x3) ===")

cols = ["температура", "опалювальні_дні", "витрати_на_опалення"]
df_three = climate[cols]

corr_pearson = df_three.corr(method="pearson")
corr_spearman = df_three.corr(method="spearman")

print("\nКореляційна матриця (Пірсон):")
print(corr_pearson.round(4))

print("\nКореляційна матриця (Спірмен):")
print(corr_spearman.round(4))

task3_text = """
[Висновки до Завдання 3]
- Найсильніша кореляція: між змінними 'опалювальні_дні' та 'витрати_на_опалення' (r > 0.98).
  Це повністю очікувано, оскільки витрати обчислюються прямою функціональною залежністю
  cost = 0.15 * heating_days + шум.
- Найслабша (найменш від'ємна) кореляція: між 'температура' та 'витрати_на_опалення' (або 'опалювальні_дні'),
  хоча вона все одно є екстремально високою по модулю (|r| > 0.95).
- Перехід до Спірмена не змінює висновків: абсолютні значення залишаються близькими до 1.0,
  зберігається той самий характер та ранг зв'язків.
"""
print(task3_text)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Кореляція і причинність
# ------------------------------------------------------------------------------
print("=== Завдання 4: Кореляція і причинність ===")
task4_text = """
У цьому конкретному випадку висока від'ємна кореляція відображає ПРЯМИЙ ПРИЧИННО-НАСЛІДКОВИЙ ЗВ'ЯЗОК,
визначений механікою генерації даних. Змінні 'опалювальні_дні' та 'витрати_на_опалення' обчислені безпосередньо
за математичною формулою від 'температура'. Зменшення температури фізично спричиняє зростання
потреби в опаленні та, як наслідок, витрат. На відміну від класичного прикладу 'морозиво й утоплення'
(де спільною причиною / конфаундером є спекотна літня погода), тут немає прихованого третього фактора:
температура виступає прямою незалежною змінною, що визначає витрати.
"""
print(task4_text)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. Перевірка на нелінійність і викиди
# ------------------------------------------------------------------------------
print("=== Завдання 5: Перевірка впливу викиду ===")

climate_outlier = climate.copy()
# Додаємо аномальний рядок: січень із витратами, збільшеними у 5 разів ("аварія на котельній")
outlier_idx = 0  # Січень
climate_outlier.loc[outlier_idx, "витрати_на_опалення"] *= 5.0
outlier_cost = climate_outlier.loc[outlier_idx, "витрати_на_опалення"]

r_out = pearsonr(climate_outlier["температура"], climate_outlier["витрати_на_опалення"])
rho_out = spearmanr(climate_outlier["температура"], climate_outlier["витрати_на_опалення"])

print(f"Змінено січень (індекс 0): витрати зросли до {outlier_cost:.1f} тис. грн.")
print(f"Новий Пірсон (r_out):   {r_out.statistic:.4f} (початковий: {r_stat:.4f})")
print(f"Новий Спірмен (rho_out): {rho_out.statistic:.4f} (початковий: {rho_stat:.4f})")

task5_text = f"""
[Порівняння впливу викиду]
- Зміна r (Пірсон): з {r_stat:.4f} до {r_out.statistic:.4f} (абсолютне відхилення: {abs(r_stat - r_out.statistic):.4f})
- Зміна rho (Спірмен): з {rho_stat:.4f} до {rho_out.statistic:.4f} (абсолютне відхилення: {abs(rho_stat - rho_out.statistic):.4f})

ПОЯСНЕННЯ МЕХАНІКИ:
Коефіцієнт Пірсона зазнав більшого викривлення (або помітнішої зміни), оскільки він обчислюється
на основі точних числових значень та їхніх відхилень від середнього у квадраті. Одиничний екстремальний
викид суттєво зміщує коваріацію та стандартне відхилення.
Коефіцієнт Спірмена є стійким (робастним), бо він переводить значення у ранги. Аномально високе значення
просто отримує найвищий ранг (12-й), як і отримало б будь-яке інше найбільше значення,
тому рангова структура та статистика Спірмена залишаються максимально стабільними.
"""
print(task5_text)

=== ПРАКТИКА 16: Варіант 6 (Чернігів) ===
   місто місяць  температура  опалювальні_дні  витрати_на_опалення
Чернігів    Січ           -5               23                  4.2
Чернігів    Лют           -4               22                  2.4
Чернігів    Бер            1               13                  2.9
Чернігів    Кві            9               11                  0.1
Чернігів    Тра           15                5                  0.3
Чернігів    Чер           18                4                  1.2
Чернігів    Лип           20                0                  1.2
Чернігів    Сер           19                0                  0.1
Чернігів    Вер           13                5                  0.6
Чернігів    Жов            7               12                  2.6
Чернігів    Лис            1               18                  4.5
Чернігів    Гру           -3               19                  2.4

=== Завдання 1: Діаграма розсіювання ===
Графік збережено як 'scatter_plot_v6.png'.
Оп

# Відповіді на контрольні питання
**Практична робота №15: Критерії узгодженості й таблиці спряженості: критерій хі-квадрат**
**Варіант 6:** м. Чернігів ($\text{base\_temp} = 8.0 \text{ °C}$, $\text{amplitude} = 13 \text{ °C}$)

---

### 1. Чи можете пояснити механіку статистики $\chi^2$ — чому в ній саме квадрат відхилення й саме ділення на очікувану частоту?

Формула статистики хі-квадрат має вигляд:
$$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$

* **Квадрат відхилення $(O_i - E_i)^2$:**
  1. *Компенсація знаків:* Відхилення спостережуваних частот від очікуваних можуть бути як додатними ($O_i > E_i$), так і від'ємними ($O_i < E_i$). Без піднесення до квадрата їхня сума завжди дорівнювала б нулю ($\sum (O_i - E_i) = 0$).
  2. *Штраф за великі відхилення:* Піднесення до квадрата підсилює вагомість великих розбіжностей між теорією та фактом.
* **Ділення на очікувану частоту $E_i$:**
  Нормалізує відхилення залежно від масштабу клітинки. Відхилення на 10 одиниць у клітинці, де очікувалося 1000 спостережень ($10^2 / 1000 = 0.1$), є незначним випадковим шумом. Проте таке ж відхилення на 10 одиниць у клітинці з очікуваним значенням 10 ($10^2 / 10 = 10$) є критичним відхиленням від теорії.

---

### 2. Чи можете пояснити, як обчислюється очікувана частота клітинки таблиці спряженості ($\text{рядок} \times \text{стовпець} / \text{загальна сума}$) і чому саме ця формула коректна під припущенням незалежності?

* **Теоретичне обґрунтування:**
  Відповідно до теорії ймовірностей, дві події $A$ (належність до рядка $i$) та $B$ (належність до стовпця $j$) є **незалежними**, якщо ймовірність їхнього сумісного настання дорівнює добутку їхніх крайових (маргінальних) ймовірностей:
  $$P(A \cap B) = P(A) \cdot P(B)$$

* **Математичний вивід:**
  1. Оцінка ймовірності потрапити в рядок $i$: $P(A) = \frac{R_i}{N}$ (де $R_i$ — сума елементів рядка $i$, $N$ — загальна сума).
  2. Оцінка ймовірності потрапити в стовпець $j$: $P(B) = \frac{C_j}{N}$ (де $C_j$ — сума елементів стовпця $j$).
  3. Очікувана ймовірність клітинки $(i, j)$: $P(A \cap B) = \frac{R_i}{N} \cdot \frac{C_j}{N}$.
  4. Очікувана абсолютна частота клітинки ($E_{ij}$):
     $$E_{ij} = N \cdot P(A \cap B) = N \cdot \frac{R_i}{N} \cdot \frac{C_j}{N} = \frac{R_i \cdot C_j}{N}$$

---

### 3. Чи розумієте різницю між "p-value велике — немає підстав відхилити $H_0$" і "p-value велике — доведено, що змінні незалежні"?

* **Неможливість «доведення» нульової гіпотези:**
  У статистиці високе $p$-value ($p > \alpha$) **не доводить** істинність нульової гіпотези. Воно лише вказує на те, що наявні вибіркові дані не суперечать нульовій гіпотезі і спостережувані розбіжності можуть бути пояснені випадковими вибірковими коливаннями.
* **Причина:**
  Високе $p$-value може бути результатом малого обсягу вибірки (низька статистична потужність тесту). Наприклад, реальний слабкий зв'язок може бути невиявленим на 10 спостереженнях, але стане статистично значущим на 1000 спостереженнях.
* **Правильне формулювання:**
  «Дані узгоджуються з нульовою гіпотезою» або «Немає достатніх статистичних підстав стверджувати про наявність залежності».

---

### 4. Що робити, якщо очікувана частота в якійсь клітинці таблиці спряженості менша за 5, і чому це проблема?

* **У чому полягає проблема:**
  Критерій $\chi^2$ є асимптотичним — статистика тесту прямує до теоретичного розподілу $\chi^2$ лише при великих обсягах вибірки. Коли $E_i < 5$, ділення на мале число в формулі $\frac{(O_i - E_i)^2}{E_i}$ робить статистику надто чутливою до випадкових коливань, апроксимація розподілу порушується, і обчислене $p$-value стає неточним (зазвичай штучно заниженим).

* **Методи вирішення:**
  1. **Об'єднання категорій (Category Pooling):** Логічне групування суміжних або рідкісних категорій для збільшення очікуваних частот (як зроблено в Завданні 3, де об'єднали «холодніше» і «звичайно»).
  2. **Точний тест Фішера (Fisher's Exact Test):** Застосовується для таблиць $2 \times 2$ при малих частотах (`scipy.stats.fisher_exact`).
  3. **Збільшення обсягу вибірки:** Збір додаткових спостережень.